<!--
Projeto : Agente Especialista em GRC - RAG + LLM local sobre o COSO ERM 2017
Arquivo : main.ipynb
-->
# Agente Especialista em GRC &ndash; demonstração

**Universidade Presbiteriana Mackenzie** &ndash; Faculdade de Computação e Informática
Sistemas de Informação &ndash; Inteligência Artificial (7ºK, noite) &ndash; Prof. Dr. Leandro Zerbinatti

| Integrante | RA | E-mail |
|---|---|---|
| Guilherme Martins Silva | 10417140 | 10417140@mackenzista.com.br |
| Enzo Ribeiro | 10418262 | 10418262@mackenzista.com.br |
| Pedro Nogueira Ribeiro | 10324924 | 10324924@mackenzista.com.br |

**Síntese:** notebook principal do agente. Inicializa o pipeline completo (re-ranker, memória
conversacional, banco vetorial do COSO ERM 2017 e Llama 3.2 3B Instruct em 4 bits) e responde a
perguntas em linguagem natural, mostrando a perplexidade e os trechos (chunks) usados como fonte.

**Histórico de alterações**

| Data | Autor | Descrição |
|---|---|---|
| 2026-02-26 | Guilherme Martins Silva | Versão inicial (instalação, inicialização e uma pergunta de teste). |
| 2026-11-15 | Enzo Ribeiro | Cabeçalho padronizado; `pip install` substituído pelo `requirements.txt`; escolha do modelo por variável de ambiente e dos modelos auxiliares na CPU em GPUs com menos de 8 GB; demonstração da memória conversacional e da abstenção fora do escopo. |

## Como executar

1. `pip install -r requirements.txt` (veja o `README.md`).
2. Modelo gerador: o repositório oficial `meta-llama/Llama-3.2-3B-Instruct` exige aceite da licença e
   `hf auth login`. Sem isso, defina `LLAMA_MODEL_ID=unsloth/Llama-3.2-3B-Instruct`,
   um espelho público com os mesmos pesos.
3. Execute as células em ordem a partir da raiz do repositório.

In [1]:
import os
import warnings

import torch

from llama.call_llama import CallLlama

warnings.filterwarnings("ignore")

MODEL_ID = os.environ.get("LLAMA_MODEL_ID", "meta-llama/Llama-3.2-3B-Instruct")

# Com menos de 8 GB de VRAM, embeddings e re-ranker rodam na CPU e a GPU fica só para o Llama
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0
AUX_DEVICE = None if vram_gb >= 8 else "cpu"
print(f"Modelo: {MODEL_ID} | VRAM: {vram_gb:.1f} GB | modelos auxiliares em: {AUX_DEVICE or 'GPU'}")

/home/enzoribeiroz/VSCode/mackenzie/ai/ai_lab_mack/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Modelo: unsloth/Llama-3.2-3B-Instruct | VRAM: 6.1 GB | modelos auxiliares em: cpu


In [2]:
llm = CallLlama(model_id=MODEL_ID, aux_device=AUX_DEVICE, verbose=False)

🚀 Inicializando call_llama...


Progresso Geral:   0%|          | 0/5

🔍 Carregando sistema de Re Rank:   0%|          | 0/5

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 393/393 [00:00<00:00, 15497.51it/s]


🔍 Carregando sistema de Re Rank:  20%|██        | 1/5

... (50 outputs left out)


⚙️ Carregando modelo Llama local:  80%|████████  | 4/5

📊 Inicializando Answer Evaluation:  80%|████████  | 4/5

📊 Inicializando Answer Evaluation: 100%|██████████| 5/5


✅ call_llama inicializado com sucesso!


### 1. Pergunta objetiva sobre o framework

In [3]:
resposta = llm.invoke(
    user="Quantos e quais são os componentes que estruturam o framework (ERM)?",
    seed=42
)
print(resposta)
print("\nFontes (chunks):", llm.last_metrics["sources"])


🔍 Recuperando memórias relevantes para: 'Quantos e quais são os componentes que estruturam ...'
🎯 Memórias encontradas:
 - dist 0.1866 (INCLUÍDA): USER: Quantos e quais sãos componentes que estruturam o framework (ERM)?
LLM:  O...


[transformers] Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.



💾 Adicionando memória: USER: Quantos e quais são os componentes que estruturam o fr...


🟩 Memória adicionada. ID: a23f42ec-692a-4b8e-bf1d-0820bcd3b05a
📊 Total de memórias agora: 2
📂 Arquivos no DB: ['f0237d3f-40dd-4804-aef9-b71168363bb7', 'chroma.sqlite3']
  - chroma.sqlite3: 225280 bytes
🧼 Limpando e purificando memórias...
📊 Total de memórias antes da limpeza: 2
🧹 Após deduplicação: 1 memórias
✂️ Memórias após limite: 1
🔄 Reescrevendo coleção...
🗑️ Deletados 2 registros antigos
✅ Coleção reescrita. Total final: 1

Mean logprob: -0.0323
Perplexity: 1.0328
De acordo com o contexto fornecido, o Framework de Gestão de Risco (GRC) é composto por 5 componentes principais:

1. **Governança e Cultura**: Estabelece a tonalidade da organização e estabelece as responsabilidades de supervisão para a gestão de risco.
2. **Estratégia e Definição de Objetivos**: Integra a gestão de risco com a estratégia e a definição de objetivos, estabelecendo a apetite de risco e alinhando a estratégia com os objetivos da organização.
3. **Desempenho**: Identifica e avalia os riscos que podem afeta

### 2. Pergunta de acompanhamento (usa a memória conversacional)

In [4]:
resposta = llm.invoke(
    user="Dentre esses componentes, qual trata da definição do apetite a risco e por quê?",
    seed=42
)
print(resposta)


🔍 Recuperando memórias relevantes para: 'Dentre esses componentes, qual trata da definição ...'
🎯 Memórias encontradas:
 - dist 0.2815 (INCLUÍDA): USER: Quantos e quais sãos componentes que estruturam o framework (ERM)?
LLM:  O...


[transformers] Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



💾 Adicionando memória: USER: Dentre esses componentes, qual trata da definição do a...
🟩 Memória adicionada. ID: 9c90f667-a390-4103-aee6-1e298e6ff0c8
📊 Total de memórias agora: 2
📂 Arquivos no DB: ['f0237d3f-40dd-4804-aef9-b71168363bb7', 'chroma.sqlite3']
  - chroma.sqlite3: 258048 bytes
🧼 Limpando e purificando memórias...
📊 Total de memórias antes da limpeza: 2
🧹 Após deduplicação: 2 memórias
✂️ Memórias após limite: 2

Mean logprob: -0.1895
Perplexity: 1.2087
O componente que trata da definição do apetite a risco é o "Desempenho" (Componente 3 do Framework de Gestão de Risco, ou ERM).

O desempenho é responsável por identificar e avaliar os riscos que podem afetar a atuação da estratégia e os objetivos da organização, priorizando os riscos de acordo com a apetite de risco.

A definição do apetite a risco é um aspecto fundamental do desempenho, pois permite que a organização estabeleça um nível de tolerância para os riscos que ela está disposta a assumir em busca de valor. Isso é fe

### 3. Pergunta fora do escopo do COSO ERM (o agente deve se abster)

In [5]:
resposta = llm.invoke(user="Qual é a taxa Selic definida na última reunião do Copom?", seed=42)
print(resposta)


🔍 Recuperando memórias relevantes para: 'Qual é a taxa Selic definida na última reunião do ...'
🎯 Memórias encontradas:
 - dist 0.3764 (INCLUÍDA): USER: Dentre esses componentes, qual trata da definição do apetite a risco e por...
 - dist 0.3806 (INCLUÍDA): USER: Quantos e quais sãos componentes que estruturam o framework (ERM)?
LLM:  O...


[transformers] Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



💾 Adicionando memória: USER: Qual é a taxa Selic definida na última reunião do Copo...
🟩 Memória adicionada. ID: 2e286d16-d61d-4213-83dd-285334a7bd09
📊 Total de memórias agora: 3
📂 Arquivos no DB: ['f0237d3f-40dd-4804-aef9-b71168363bb7', 'chroma.sqlite3']
  - chroma.sqlite3: 266240 bytes
🧼 Limpando e purificando memórias...
📊 Total de memórias antes da limpeza: 3
🧹 Após deduplicação: 3 memórias
✂️ Memórias após limite: 3

Mean logprob: -0.2004
Perplexity: 1.2219
Infelizmente, não tenho acesso a informações em tempo real ou atualizações após a data do contexto fornecido (dezembro de 2023). A taxa Selic é definida pelo Banco Central do Brasil e é ajustada pela Comissão de Política Monetária do Banco Central (Copom) em reuniões regulares.

Para obter a informação mais atualizada sobre a taxa Selic, recomendo verificar o site do Banco Central do Brasil ou notícias financeiras recentes.
